[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C49_Encoder_Seq2Seq_Course/05_encoder_today/05_encoder_today.ipynb)

# 05 · 今天还要不要 encoder（成本-精度前沿、蒸馏、两阶段检索）

目标：把 **推理成本模型 → 温度蒸馏与 T² 因子 → 双塔 vs 交叉编码 → 两阶段最优 k → 选型决策** 全部算清楚。

路线：FLOPs 与延迟账 → 软标签的信息量 → 蒸馏梯度的 T² 验证 → 各向异性演示 →
InfoNCE 与 hard negatives → 召回-精排成本-精度前沿 → 选型决策树 → ✏️ 练习 → 📖 答案 → 🧪 真实选型胶囊。

> 心智模型：**「用便宜的模型过滤，用贵的模型处理剩下的」是这个领域最通用、最被低估的架构模式。**

## 1 · 推理成本：把「便宜」变成数字

`FLOPs ≈ 2 × 参数量 × 处理的 token 数`。关键差异在**处理的 token 数**：
encoder 编码一次；LLM 要 prefill + 每生成一个 token 再跑一遍全部权重。

In [ ]:
import numpy as np, math
rng = np.random.default_rng(0)

def forward_flops(params, n_tokens):
    return 2 * params * n_tokens

def encoder_cost(params, n_in):
    return forward_flops(params, n_in)

def llm_cost(params, n_in, n_out):
    prefill = forward_flops(params, n_in)
    decode  = sum(forward_flops(params, 1) for _ in range(n_out))   # 每步都要读全部权重
    return prefill + decode

N_IN = 128
configs = [
    ('DistilBERT (6L)',            'enc',  66e6,  0),
    ('BERT-base',                  'enc', 110e6,  0),
    ('DeBERTa-v3-base',            'enc', 184e6,  0),
    ('Llama-8B  输出5 token',      'llm',   8e9,  5),
    ('Llama-8B  输出100 token',    'llm',   8e9, 100),
    ('Llama-70B 输出100 token',    'llm',  70e9, 100),
]
base = encoder_cost(110e6, N_IN)
print(f"{'方案':<28s} {'FLOPs':>12s} {'相对BERT-base':>14s}")
rows = []
for name, kind, p, n_out in configs:
    f = encoder_cost(p, N_IN) if kind == 'enc' else llm_cost(p, N_IN, n_out)
    rows.append((name, f))
    print(f'{name:<28s} {f:>12.3e} {f/base:>13.1f}×')

f_bert = dict(rows)['BERT-base']
f_llm5 = dict(rows)['Llama-8B  输出5 token']
f_llm100 = dict(rows)['Llama-8B  输出100 token']
assert f_llm5 / f_bert > 50, 'LLM 即使只输出 5 token 也贵 50 倍以上'
assert f_llm100 > f_llm5, '输出越长越贵'
print(f'\n✅ 同样做二分类：LLM(输出5tok) 比 BERT-base 贵 {f_llm5/f_bert:.0f}×，'
      f'带解释(100tok) 贵 {f_llm100/f_bert:.0f}×')

### 延迟与「能不能跑 CPU」——比 FLOPs 更硬的约束

LLM 的 decode 是 **memory-bound**：每生成一个 token 都要把全部权重读一遍。

In [ ]:
def decode_latency_ms(params_bytes, bandwidth_gbps, n_out):
    '''memory-bound 下界：每步至少要读一遍权重。'''
    per_step_s = params_bytes / (bandwidth_gbps * 1e9)
    return per_step_s * n_out * 1000

def encoder_latency_ms(flops, tflops):
    return flops / (tflops * 1e12) * 1000

GPU_BW_GBPS, GPU_TFLOPS = 2000, 300      # H100 量级
CPU_TFLOPS = 0.5                          # 现代服务器 CPU 单路量级

print(f"{'方案':<28s} {'GPU延迟(ms)':>12s} {'CPU延迟(ms)':>12s} {'能跑CPU?':>9s}")
for name, kind, p, n_out in configs:
    if kind == 'enc':
        f = encoder_cost(p, N_IN)
        g, c = encoder_latency_ms(f, GPU_TFLOPS), encoder_latency_ms(f, CPU_TFLOPS)
    else:
        g = decode_latency_ms(p * 2, GPU_BW_GBPS, n_out) + encoder_latency_ms(forward_flops(p, N_IN), GPU_TFLOPS)
        c = decode_latency_ms(p * 2, 50, n_out)          # CPU 内存带宽约 50 GB/s
    ok = '✅' if c < 200 else '❌'
    print(f'{name:<28s} {g:>12.1f} {c:>12.1f} {ok:>9s}')

lat_bert_cpu = encoder_latency_ms(encoder_cost(110e6, N_IN), CPU_TFLOPS)
lat_llm_cpu = decode_latency_ms(8e9 * 2, 50, 100)
assert lat_bert_cpu < 200, 'BERT-base 在 CPU 上可用'
assert lat_llm_cpu > 5000, '8B 模型在 CPU 上生成 100 token 要几十秒 —— 实用性为零'
print(f'\n✅ 「不需要 GPU」本身就是巨大的成本与运维差异：')
print(f'   BERT-base CPU {lat_bert_cpu:.0f}ms 可用 | Llama-8B CPU {lat_llm_cpu/1000:.0f}s 不可用')
print('   选 encoder 意味着 C48 里整个 GPU 调度与成本的问题都可以跳过。')

## 2 · 蒸馏：软标签为什么比硬标签强

软标签携带**类间相似结构**（dark knowledge），硬标签只有 log2(C) bit。

In [ ]:
def softmax_T(logits, T=1.0):
    z = logits / T
    z = z - z.max()
    e = np.exp(z); return e / e.sum()

teacher_logits = np.array([1.0, 6.0, 3.5, -2.0])       # 狗 猫 虎 汽车
CLASSES = ['狗', '猫', '虎', '汽车']
print(f"{'温度':>5s} " + ' '.join(f'{c:>7s}' for c in CLASSES))
for T in [1.0, 2.0, 4.0, 8.0]:
    p = softmax_T(teacher_logits, T)
    print(f'{T:>5.1f} ' + ' '.join(f'{x:>7.3f}' for x in p))

hard = np.zeros(4); hard[1] = 1.0
def entropy(p): return -np.sum(p * np.log(p + 1e-12))
print(f'\n硬标签熵: {entropy(hard):.3f} nat（0 = 只说「是猫」）')
for T in [1.0, 4.0]:
    print(f'软标签熵 (T={T}): {entropy(softmax_T(teacher_logits, T)):.3f} nat')

p1, p4 = softmax_T(teacher_logits, 1.0), softmax_T(teacher_logits, 4.0)
assert entropy(p4) > entropy(p1) > entropy(hard), '温度越高分布越平滑、信息越丰富'
assert p1[2] > p1[0] > p1[3], '软标签保留了「虎 > 狗 > 汽车」的相似结构'
print('\n✅ 软标签告诉 student「这个样本有点像虎」—— 这条信息在 one-hot 里完全丢失。')

### T² 因子：忘了它等于没做蒸馏

软标签交叉熵对 student logits 的梯度按 `1/T²` 衰减。**必须乘 T² 补回来。**

In [ ]:
def kd_grad_scale(teacher_logits, student_logits, T, use_T2=True):
    '''软标签 KL 项对 student logits 的梯度范数。'''
    pt = softmax_T(teacher_logits, T)
    ps = softmax_T(student_logits, T)
    grad = (ps - pt) / T                       # d KL / d student_logits
    if use_T2:
        grad = grad * (T ** 2)
    return float(np.abs(grad).sum())

student_logits = np.array([0.5, 2.0, 1.0, 0.0])
hard_grad = float(np.abs(softmax_T(student_logits, 1.0) - hard).sum())
print(f'硬标签项梯度范数: {hard_grad:.4f}\n')
print(f"{'T':>5s} {'不乘T² ':>12s} {'乘T² ':>10s} {'不乘时相对硬标签':>18s}")
for T in [1.0, 2.0, 4.0, 8.0]:
    g_no = kd_grad_scale(teacher_logits, student_logits, T, use_T2=False)
    g_yes = kd_grad_scale(teacher_logits, student_logits, T, use_T2=True)
    print(f'{T:>5.1f} {g_no:>12.5f} {g_yes:>10.4f} {g_no/hard_grad:>17.1%}')

g_no_4 = kd_grad_scale(teacher_logits, student_logits, 4.0, use_T2=False)
g_no_1 = kd_grad_scale(teacher_logits, student_logits, 1.0, use_T2=False)
assert g_no_4 < g_no_1 / 5, 'T=4 时不乘 T²，梯度衰减一个量级以上'
g_yes_4 = kd_grad_scale(teacher_logits, student_logits, 4.0, use_T2=True)
assert g_yes_4 > g_no_4 * 10, 'T² 把梯度补回来'
print(f'\n⚠️  T=4 且忘乘 T²：蒸馏项梯度只有硬标签项的 {g_no_4/hard_grad:.1%} —— 等于没做蒸馏。')
print('✅ T² 因子已验证')

### 数据蒸馏：LLM → encoder 唯一可行的路

两者输出空间完全不同（token 分布 vs 类别分布），**只能走数据蒸馏**：
让 LLM 打标签（可带置信度当软标签），再当普通监督数据用。

In [ ]:
def simulate_data_distillation(n_unlabeled, teacher_acc, student_capacity=0.98, seed=0):
    '''student 的上限 = teacher 准确率 × student 容量系数。'''
    r = np.random.default_rng(seed)
    true_y = r.integers(0, 2, size=n_unlabeled)
    teacher_y = np.where(r.random(n_unlabeled) < teacher_acc, true_y, 1 - true_y)
    # student 在 teacher 标签上学到 student_capacity 的一致性
    student_y = np.where(r.random(n_unlabeled) < student_capacity, teacher_y, 1 - teacher_y)
    return (student_y == true_y).mean(), (teacher_y == true_y).mean()

print(f"{'teacher 准确率':>14s} {'student 准确率':>14s} {'损失':>7s}")
for ta in [0.80, 0.86, 0.92, 0.96]:
    sa, real_ta = simulate_data_distillation(20000, ta)
    print(f'{real_ta:>14.1%} {sa:>14.1%} {real_ta - sa:>7.1%}')

sa96, ta96 = simulate_data_distillation(20000, 0.96)
sa80, ta80 = simulate_data_distillation(20000, 0.80)
assert sa96 > sa80, 'teacher 越强，student 越强'
assert sa96 < ta96, 'student 一般不超过 teacher（除非有额外的真标注锚定）'
print('\n✅ **student 的上限由 teacher 决定** —— 这也意味着 teacher 的系统性偏差会被完整继承。')
print('   实践建议：留一小份人工标注做验证集，用来检测蒸馏是否继承了 LLM 的偏差（C03/C10）。')

## 3 · 句嵌入：为什么不能直接用预训练 BERT 的向量

**各向异性**：BERT 表示挤在一个狭窄锥体里，任意两句的余弦相似度都很高，失去区分度。

In [ ]:
D = 64

def make_anisotropic(n, d, cone_strength, seed=0):
    '''cone_strength 越大，向量越集中在一个主方向上（各向异性越强）。'''
    r = np.random.default_rng(seed)
    main = r.normal(size=d); main /= np.linalg.norm(main)
    X = r.normal(size=(n, d))
    X = X / np.linalg.norm(X, axis=1, keepdims=True)     # 先归一化噪声，再叠加主方向
    X = X + cone_strength * main
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def mean_pairwise_cos(X, n_pairs=3000, seed=0):
    r = np.random.default_rng(seed)
    i = r.integers(0, len(X), n_pairs); j = r.integers(0, len(X), n_pairs)
    m = i != j
    return float((X[i[m]] * X[j[m]]).sum(1).mean())

print(f"{'各向异性强度':>12s} {'平均两两余弦':>13s} {'相似度动态范围':>15s}")
for cs in [0.0, 0.8, 1.5, 3.0]:
    X = make_anisotropic(600, D, cs)
    r_ = np.random.default_rng(1)
    i, j = r_.integers(0, 600, 3000), r_.integers(0, 600, 3000)
    cos = (X[i] * X[j]).sum(1)
    print(f'{cs:>12.1f} {mean_pairwise_cos(X):>13.3f} {cos.max()-cos.min():>15.3f}')

X_iso, X_aniso = make_anisotropic(600, D, 0.0), make_anisotropic(600, D, 3.0)
assert mean_pairwise_cos(X_aniso) > 0.7, '强各向异性下任意两句相似度都很高'
assert abs(mean_pairwise_cos(X_iso)) < 0.1, '各向同性时随机两句应接近正交'
print('\n✅ 各向异性让「任意两句都很像」—— 余弦相似度失去区分度。')
print('   这就是 Reimers & Gurevych 发现「原始 BERT 句向量不如 GloVe 平均」的原因之一。')

In [ ]:
def whiten(X):
    '''白化：去均值 + 协方差归一化，缓解各向异性。'''
    mu = X.mean(0)
    Xc = X - mu
    cov = Xc.T @ Xc / len(X)
    U, S, _ = np.linalg.svd(cov)
    W = U @ np.diag(1.0 / np.sqrt(S + 1e-8))
    Y = Xc @ W
    return Y / np.linalg.norm(Y, axis=1, keepdims=True)

X_w = whiten(X_aniso)
print(f'白化前 平均余弦 {mean_pairwise_cos(X_aniso):>6.3f}')
print(f'白化后 平均余弦 {mean_pairwise_cos(X_w):>6.3f}')
assert abs(mean_pairwise_cos(X_w)) < abs(mean_pairwise_cos(X_aniso)) / 3
print('✅ 白化是零训练成本的缓解手段；更根本的解法是对比学习（SBERT / SimCSE）')

### InfoNCE 与 hard negatives：负例设计再次决定一切

In [ ]:
def info_nce(q, d_pos, d_negs, tau=0.05):
    '''InfoNCE 损失。'''
    s_pos = float(q @ d_pos) / tau
    s_negs = np.array([float(q @ d) for d in d_negs]) / tau
    all_s = np.concatenate([[s_pos], s_negs])
    all_s = all_s - all_s.max()
    return -(all_s[0] - np.log(np.exp(all_s).sum()))

r = np.random.default_rng(3)
q = r.normal(size=D); q /= np.linalg.norm(q)
d_pos = q + r.normal(size=D) * 0.3; d_pos /= np.linalg.norm(d_pos)

def make_negs(kind, n=16):
    negs = []
    for _ in range(n):
        if kind == 'random':
            v = r.normal(size=D)                       # 随机负例：与 q 几乎正交
        else:
            v = q + r.normal(size=D) * 0.45            # hard negative：很像但不是
        negs.append(v / np.linalg.norm(v))
    return negs

for kind in ['random', 'hard']:
    negs = make_negs(kind)
    loss = info_nce(q, d_pos, negs)
    sim = np.mean([float(q @ n_) for n_ in negs])
    print(f'{kind:>7s} negatives: 平均相似度 {sim:>6.3f} | InfoNCE 损失 {loss:>6.3f}')

loss_rand = info_nce(q, d_pos, make_negs('random'))
loss_hard = info_nce(q, d_pos, make_negs('hard'))
assert loss_hard > loss_rand, 'hard negatives 让任务更难 -> 损失更大 -> 梯度信号更有用'
print('\n✅ 随机负例太容易区分（主题都不同），模型学不到细粒度。')
print('   这与模块 01 的 NSP 失败、SOP 成功是**同一条原理的第三次体现**：')
print('   **负例设计决定任务难度，也决定模型学到什么。**')

## 4 · 两阶段检索：成本-精度前沿与最优 k

召回只需「不漏掉」，精排只处理少量候选。**最优 k 落在 recall 曲线的拐点。**

In [ ]:
N_DOCS = 1_000_000

def recall_at_k(k, saturation=30.0, ceiling=0.99):
    '''双塔召回的 recall@k：随 k 快速饱和。'''
    return ceiling * (1 - math.exp(-k / saturation))

def two_stage_quality(k, cross_precision=0.95):
    return recall_at_k(k) * cross_precision

def two_stage_cost(k, bi_flops=2.8e10, cross_flops=2.8e10, ann_flops=1e8):
    '''1 次 query 编码 + ANN 检索 + k 次交叉编码。'''
    return bi_flops + ann_flops + k * cross_flops

print(f"{'k':>6s} {'recall@k':>9s} {'端到端质量':>10s} {'成本(FLOPs)':>13s} {'质量/成本':>11s}")
best_k, best_ratio = None, -1
for k in [1, 10, 50, 100, 200, 500, 1000]:
    q_, c_ = two_stage_quality(k), two_stage_cost(k)
    ratio = q_ / (c_ / 1e10)
    if ratio > best_ratio: best_ratio, best_k = ratio, k
    print(f'{k:>6d} {recall_at_k(k):>9.3f} {q_:>10.3f} {c_:>13.3e} {ratio:>11.4f}')

# 全量交叉编码（方案 A）：完全不可行
full_cross = N_DOCS * 2.8e10
print(f'\n全量交叉编码 {N_DOCS:,} 文档: {full_cross:.3e} FLOPs')
print(f'两阶段 (k=100):              {two_stage_cost(100):.3e} FLOPs')
print(f'节省 {full_cross / two_stage_cost(100):,.0f}×')
assert two_stage_cost(100) < full_cross / 1000, '两阶段应便宜 3 个数量级以上'
assert two_stage_quality(100) > 0.9, '且质量仍在 0.9 以上'
# recall 饱和：k 从 100 到 1000 提升很小，成本却涨 10 倍
gain = recall_at_k(1000) - recall_at_k(100)
cost_up = two_stage_cost(1000) / two_stage_cost(100)
assert gain < 0.20 and cost_up > 5, f'k 100->1000: recall 只涨 {gain:.3f} 但成本涨 {cost_up:.1f}×'
print(f'\n✅ k: 100->1000 recall 只涨 {gain:.3f}，成本涨 {cost_up:.1f}× —— 最优 k 在拐点附近（50~200）')

### 双塔的可预计算性：这才是它不可替代的原因

In [ ]:
def bi_encoder_online_cost(n_docs, query_flops=2.8e10, ann_flops=1e8):
    '''文档向量**离线**预计算，在线只编码 query + ANN。与 n_docs 几乎无关！'''
    return query_flops + ann_flops

def cross_encoder_online_cost(n_docs, pair_flops=2.8e10):
    return n_docs * pair_flops

print(f"{'语料规模':>12s} {'双塔在线':>13s} {'交叉编码在线':>15s} {'倍数':>12s}")
for n in [1_000, 100_000, 10_000_000]:
    b, c = bi_encoder_online_cost(n), cross_encoder_online_cost(n)
    print(f'{n:>12,d} {b:>13.2e} {c:>15.2e} {c/b:>11,.0f}×')

b1k, b10m = bi_encoder_online_cost(1_000), bi_encoder_online_cost(10_000_000)
assert abs(b1k - b10m) < 1e-9, '**双塔的在线成本与语料规模无关** —— 这是它的核心价值'
print('\n✅ 双塔的在线成本**与语料规模无关**（文档向量离线算好）。')
print('   交叉编码则随语料线性增长。这就是「交互带来精度，独立带来可预计算」。')

## ✏️ 练习 1：蒸馏损失

实现 `distillation_loss(teacher_logits, student_logits, true_label, T=4.0, alpha=0.3)`：
返回 `alpha * CE(hard) + (1-alpha) * T² * KL(teacher_T || student_T)`。
KL 用 `Σ p_t * (log p_t - log p_s)`。

In [ ]:
def distillation_loss(teacher_logits, student_logits, true_label, T=4.0, alpha=0.3):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
tl = np.array([1.0, 6.0, 3.5, -2.0]); sl = np.array([0.5, 2.0, 1.0, 0.0])
loss = distillation_loss(tl, sl, true_label=1)
assert loss > 0
# student 完全模仿 teacher 时，KL 项应为 0
loss_perfect = distillation_loss(tl, tl, true_label=1)
kl_part = loss_perfect - 0.3 * (-math.log(softmax_T(tl, 1.0)[1]))
assert abs(kl_part) < 1e-9, 'student==teacher 时 KL 项必须为 0'
# alpha=1 时退化为纯硬标签
loss_hard_only = distillation_loss(tl, sl, 1, alpha=1.0)
assert abs(loss_hard_only - (-math.log(softmax_T(sl, 1.0)[1]))) < 1e-9
# T 越大，KL 项（乘过 T² 后）不应塌陷
l_t2 = distillation_loss(tl, sl, 1, T=2.0, alpha=0.0)
l_t8 = distillation_loss(tl, sl, 1, T=8.0, alpha=0.0)
assert l_t2 > 0 and l_t8 > 0
print(f'完整蒸馏损失 {loss:.4f} | 纯硬标签 {loss_hard_only:.4f} | student=teacher {loss_perfect:.4f}')
print('✅ 练习 1 通过')

## ✏️ 练习 2：两阶段的最优 k

实现 `optimal_k(quality_fn, cost_fn, k_candidates, min_quality)`：
在满足 `quality_fn(k) >= min_quality` 的候选里，返回**成本最低**的 k。
无解返回 `None`。

In [ ]:
def optimal_k(quality_fn, cost_fn, k_candidates, min_quality):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ks = [1, 10, 25, 50, 100, 200, 500, 1000]
k = optimal_k(two_stage_quality, two_stage_cost, ks, min_quality=0.90)
assert k is not None and two_stage_quality(k) >= 0.90
assert all(two_stage_cost(k) <= two_stage_cost(x) for x in ks if two_stage_quality(x) >= 0.90)
k_strict = optimal_k(two_stage_quality, two_stage_cost, ks, min_quality=0.94)
assert k_strict > k, '更高的质量要求需要更大的 k'
assert optimal_k(two_stage_quality, two_stage_cost, ks, min_quality=0.999) is None, '不可达时返回 None'
print(f'质量≥0.90 -> k={k} (成本 {two_stage_cost(k):.2e})')
print(f'质量≥0.94 -> k={k_strict} (成本 {two_stage_cost(k_strict):.2e})')
print('✅ 练习 2 通过：k 不是拍脑袋定的，是从质量约束反解出来的')

## ✏️ 练习 3：选型决策树

实现 `choose_architecture(output_type, n_labeled, daily_calls, latency_budget_ms)`：
按讲解里的决策树返回方案字符串。
- `output_type='free_text'` → `'generative'`
- 否则若 `n_labeled == 0` → `'llm_zero_shot'`（若 `daily_calls < 100_000`）或 `'llm_distill_to_encoder'`
- 否则若 `daily_calls >= 100_000 or latency_budget_ms < 200` → `'encoder_finetune'`
- 否则 → `'either'`

In [ ]:
def choose_architecture(output_type, n_labeled, daily_calls, latency_budget_ms):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert choose_architecture('free_text', 5000, 1000, 5000) == 'generative'
assert choose_architecture('label', 0, 1000, 5000) == 'llm_zero_shot'
assert choose_architecture('label', 0, 2_000_000, 5000) == 'llm_distill_to_encoder'
assert choose_architecture('label', 5000, 2_000_000, 5000) == 'encoder_finetune'
assert choose_architecture('label', 5000, 1000, 100) == 'encoder_finetune', '低延迟要求也指向 encoder'
assert choose_architecture('label', 5000, 1000, 5000) == 'either'
cases = [('label', 0, 2_000_000, 100), ('span', 800, 500_000, 80), ('free_text', 0, 10, 60_000)]
for c in cases:
    print(f'{str(c):<38s} -> {choose_architecture(*c)}')
print('✅ 练习 3 通过：选型的第一个问题不是「哪个更准」，而是「任务落在哪一类」')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def distillation_loss(teacher_logits, student_logits, true_label, T=4.0, alpha=0.3):
    ps1 = softmax_T(student_logits, 1.0)
    hard = -math.log(ps1[true_label] + 1e-12)
    pt, ps = softmax_T(teacher_logits, T), softmax_T(student_logits, T)
    kl = float(np.sum(pt * (np.log(pt + 1e-12) - np.log(ps + 1e-12))))
    return alpha * hard + (1 - alpha) * (T ** 2) * kl

In [ ]:
# 练习 2 参考答案
def optimal_k(quality_fn, cost_fn, k_candidates, min_quality):
    feasible = [k for k in k_candidates if quality_fn(k) >= min_quality]
    return min(feasible, key=cost_fn) if feasible else None

In [ ]:
# 练习 3 参考答案
def choose_architecture(output_type, n_labeled, daily_calls, latency_budget_ms):
    if output_type == 'free_text':
        return 'generative'
    if n_labeled == 0:
        return 'llm_zero_shot' if daily_calls < 100_000 else 'llm_distill_to_encoder'
    if daily_calls >= 100_000 or latency_budget_ms < 200:
        return 'encoder_finetune'
    return 'either'

---
## 🧪 真实数据胶囊：一个内容审核系统的选型

每天 200 万次调用，P95 延迟 < 100ms，已有 5000 条人工标注。把四个方案的账全算出来。

In [ ]:
DAILY_CALLS, LATENCY_BUDGET_MS = 2_000_000, 100
GPU_USD_PER_HOUR, GPU_TFLOPS_EFF = 4.0, 150      # 有效算力（考虑利用率）
CPU_USD_PER_HOUR, CPU_TFLOPS_EFF = 0.15, 0.3

def cost_per_call(flops, usd_per_hour, tflops_eff):
    seconds = flops / (tflops_eff * 1e12)
    return seconds / 3600 * usd_per_hour

# 统一按 GPU 计价做「同口径」比较：单位算力上 GPU 比 CPU 更便宜，
# 所以 encoder 的优势**完全来自 FLOPs 少 45 倍**，而不是「换了便宜的硬件」。
options = [
    # (名称, FLOPs, 延迟ms, 精度, 用GPU?)
    ('Llama-8B zero-shot',   llm_cost(8e9, N_IN, 5),    820, 0.86, True),
    ('Llama-8B + LoRA',      llm_cost(8e9, N_IN, 5),    820, 0.93, True),
    ('DeBERTa-v3-base 微调', encoder_cost(184e6, N_IN),  35, 0.94, True),
    ('DistilBERT 蒸馏+INT8', encoder_cost(66e6, N_IN)/2, 12, 0.92, True),
]
print(f"{'方案':<24s} {'精度':>5s} {'延迟ms':>7s} {'单次$':>11s} {'日成本$':>9s} {'年成本$':>10s} {'可行':>5s}")
results = []
for name, fl, lat, acc, use_gpu in options:
    c = cost_per_call(fl, GPU_USD_PER_HOUR, GPU_TFLOPS_EFF) if use_gpu \
        else cost_per_call(fl, CPU_USD_PER_HOUR, CPU_TFLOPS_EFF)
    daily, yearly = c * DAILY_CALLS, c * DAILY_CALLS * 365
    ok = lat <= LATENCY_BUDGET_MS
    results.append((name, acc, lat, daily, yearly, ok))
    print(f'{name:<24s} {acc:>5.2f} {lat:>7d} {c:>11.8f} {daily:>9.2f} {yearly:>10,.0f} {"✅" if ok else "❌":>5s}')

feasible = [r for r in results if r[5]]
assert len(feasible) == 2, '只有两个 encoder 方案满足延迟要求'
best = max(feasible, key=lambda r: r[1])
llm_daily = results[1][3]; enc_daily = best[3]
print(f'\n可行方案里精度最高: 「{best[0]}」 精度 {best[1]:.2f}, 日成本 ${best[3]:.2f}')
assert llm_daily / enc_daily > 20, 'LLM 方案应贵一个数量级以上'
print(f'相比 LLM+LoRA（精度 0.93）: 成本低 {llm_daily/enc_daily:.0f}×，且精度还高 0.01')
print(f'年度差额: ${results[1][4]:,.0f} vs ${best[4]:,.0f}')
print('（注：这里 encoder 也按 GPU 计价，所以优势**纯粹来自 FLOPs 少 45 倍**；')
print('  若改跑 CPU，单位算力更贵但省掉整套 GPU 运维——那是另一笔账，见 C48。）')
print('\n⚠️  但表格没说的部分同样重要：')
print('   · 若只有 0 条标注 -> LLM zero-shot 的 0.86 是你**立刻**能拿到的')
print('   · 若违规类型每周在变 -> LLM 改 prompt 即可，encoder 要重标+重训')
print('   · 若需要给出违规理由 -> encoder 给不了，需要混合方案')

**🧪 胶囊练习**：实现 `hybrid_cost(daily_calls, flag_rate, cheap_flops, expensive_flops)`：
混合架构——便宜的 encoder 处理全部流量，只有被标记为「违规」的 `flag_rate` 比例
才送给 LLM 生成解释。返回 `(总FLOPs, 相对全用LLM的节省比例)`。

In [ ]:
def hybrid_cost(daily_calls, flag_rate, cheap_flops, expensive_flops):
    # TODO: total = daily_calls*cheap_flops + daily_calls*flag_rate*expensive_flops
    #       all_llm = daily_calls * expensive_flops
    #       返回 (total, 1 - total/all_llm)
    raise NotImplementedError

In [ ]:
# 自测
cheap = encoder_cost(184e6, N_IN)
expensive = llm_cost(8e9, N_IN, 100)
total, saving = hybrid_cost(DAILY_CALLS, 0.02, cheap, expensive)
print(f'全用 LLM:  {DAILY_CALLS*expensive:.3e} FLOPs/天')
print(f'混合(2%):  {total:.3e} FLOPs/天  -> 省 {saving:.1%}')
assert saving > 0.90, '只有 2% 流量走 LLM，应省 90% 以上'
# flag_rate 越高，节省越少
_, s20 = hybrid_cost(DAILY_CALLS, 0.20, cheap, expensive)
assert s20 < saving, '标记率越高，混合架构的优势越小'
print(f'若标记率 20%: 只省 {s20:.1%}')
print('\n✅ 胶囊练习通过：**「用便宜的模型过滤，用贵的模型处理剩下的」**')
print('   —— 双塔召回+交叉精排、LLM造标注+encoder上线、encoder判断+LLM解释，')
print('   都是这同一个模式。这是本领域最通用、最被低估的架构原则。')

In [ ]:
# 📖 胶囊参考答案
def hybrid_cost(daily_calls, flag_rate, cheap_flops, expensive_flops):
    total = daily_calls * cheap_flops + daily_calls * flag_rate * expensive_flops
    all_llm = daily_calls * expensive_flops
    return total, 1 - total / all_llm

---
## 🔧 旁注：真实库里这些对应什么

- **蒸馏** → `transformers` 没有内置 Trainer；标准做法是自定义 `compute_loss` 加 KL 项。DistilBERT 的训练脚本在 `examples/research_projects/distillation`。
- **句嵌入** → `sentence-transformers`：`SentenceTransformer('all-MiniLM-L6-v2')`；训练用 `MultipleNegativesRankingLoss`（就是 InfoNCE + in-batch negatives）。
- **双塔检索** → `faiss` / `hnswlib` 做 ANN；DPR 的实现在 `transformers.DPRQuestionEncoder` / `DPRContextEncoder`。
- **交叉编码精排** → `sentence-transformers.CrossEncoder`；或 `AutoModelForSequenceClassification` 直接吃句对。
- **hard negatives 挖掘** → 用当前模型检索 top-k，排除真正的正例，剩下的当 hard negatives 迭代训练。
- **INT8 量化** → `optimum.onnxruntime` 或 `bitsandbytes`；encoder 的动态量化几乎无损（C27）。

怎么把这些串成一条真实管线，见 **C50**（生态实操）与 **C11**（RAG 与检索）。

### 小结
- **成本差距是 60(参数) × 20(前向次数) ≈ 千倍量级**，且 encoder 能跑 CPU——这意味着整个 GPU 调度问题都可以跳过。
- **成本比较只在「两者都能达到精度要求」时有意义**；输出是自由文本、需要世界知识、需求天天变，这三类 encoder 不适用。
- **软标签的价值是类间相似结构**；`T²` 因子不能忘（T=4 时忘了等于没做蒸馏）。LLM→encoder 只能走**数据蒸馏**。
- **原始 BERT 的句向量不能直接用**（[CLS] 没被训练 + 各向异性 + 目标错配）；解法是白化或对比学习（SBERT/SimCSE）。
- **负例设计第三次决定一切**：NSP 的失败、SOP 的成功、hard negatives 的价值，是同一条原理。
- **双塔的在线成本与语料规模无关**——这是它不可替代的原因。两阶段的最优 k 在 recall 曲线拐点（50–200）。
- 最通用的架构模式：**用便宜的模型过滤，用贵的模型处理剩下的**。

🎓 **本课完结。** 你现在能从「注意力掩码的三种画法」推导出三种形态的全部性质，
并为一个真实任务做出有成本依据的选型。
建议的下一站：**C50**（HuggingFace 生态实操，把本课的概念变成能跑的代码）、
**C11**（RAG 与检索）、**C27**（模型压缩与量化）。